## 06 - Combine across animals and recordings

Band SNR time courses aligned on an event and averaged per condition, a per-animal window summary with paired lines, a repeated-measures ANOVA and post-hoc paired t-tests with multiple-comparison correction.

Writes `cache/<animal>/csvs/`, `cache/<animal>/<band>/`, and under `outputs/06_combine_across_animals/` the time-course and summary figures, the aligned table, per-animal values, ANOVA and post-hoc tables and `run_info.json`.

### Variables

In [ ]:
EDF_FOLDER = "../../Completed_Analyses/Emergence/raw_edfs"  # folder holding the .edf files (relative to this notebook)
RECORDINGS = [  # one dict per recording: animal_id, condition (as recorded), recording_number, edf, channel_index, events (recording-clock s)
    dict(animal_id="PEEG01M", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG01M_2026-09-10_13_30_25_export.edf", channel_index=0, events={"onset": 0}),  # 0
    dict(animal_id="PEEG01M", condition="Induction", recording_number=2, edf="EmergRev_3Perc_Induction_PEEG01M_2026-09-01_13_23_07_export.edf", channel_index=0, events={"onset": 1800}),  # 1: 2026-09-01 session
    dict(animal_id="PEEG02M", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG02M_2026-09-10_13_30_25_export.edf", channel_index=0, events={"onset": 0}),  # 2
    dict(animal_id="PEEG02M", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG02M_2026-08-27_15_49_57_export.edf", channel_index=0, events={"onset": 1800}),  # 3
    dict(animal_id="PEEG03F", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG03F_2026-09-10_13_30_25_export.edf", channel_index=1, events={"onset": 0}),  # 4: channel 1
    dict(animal_id="PEEG03F", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG03F_2026-08-27_15_49_57_export.edf", channel_index=1, events={"onset": 1800}),  # 5: channel 1
    dict(animal_id="PEEG04F", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG04F_2026-09-10_13_30_25_export.edf", channel_index=0, events={"onset": 0}),  # 6
    dict(animal_id="PEEG04F", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG04F_2026-09-15_13_46_21_export.edf", channel_index=0, events={"onset": 1860}),  # 7: onset 1860
    dict(animal_id="PEEG01M", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG01M_2026-09-10_14_37_59_export.edf", channel_index=0, events={"onset": 900}),  # 8: 0.3% alone, onset 900
    dict(animal_id="PEEG02M", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG02M_2026-08-31_14_51_28_export.edf", channel_index=0, events={"onset": 300}),  # 9: 0.3% alone, onset 300
    dict(animal_id="PEEG03F", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG03F_2026-08-31_14_51_28_export.edf", channel_index=1, events={"onset": 300}),  # 10: 0.3% alone, channel 1, onset 300 (as in all reported results)
    dict(animal_id="PEEG04F", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG04F_2026-09-10_14_37_59_export.edf", channel_index=0, events={"onset": 900}),  # 11: 0.3% alone, onset 900
    dict(animal_id="PEEG01M", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG01M_2026-08-26_15_28_31_export.edf", channel_index=0, events={"onset": 1800}),  # 12: 2026-08-26 session: a second PEEG01M Induction session
]
CONDITIONS = {  # {condition label: [(RECORDINGS index or an inline dict, channel analysed), ...]}, one entry per animal; key order = plotting order
    "AwakeControl": [(0, 0), (2, 0), (4, 1), (6, 0)],  # awake, 0% isoflurane
    # "No Induction": [(8, 0), (9, 0), (10, 1), (11, 0)],  # uncomment for the 0.3%-alone condition (a third group)
    "Induction": [(1, 0), (3, 0), (5, 1), (7, 0)],  # 0.3% after induction; add (12, 0) for a second PEEG01M session (see step 5)
}
BAND = "delta"  # band SNR to combine: a key of ecog.spectral.BAND_RATIOS or of BAND_DEFINITIONS
BAND_DEFINITIONS = {}  # extra or overriding band SNR definitions, e.g. {"beta": {"signal": (13, 24), "noise": [(8, 13)]}}; recompute after redefining an existing name
ALIGN_EVENT = "onset"  # event placed at 0 on the shared time axis (every recording needs it in its events)
BIN_S = 120  # time bin of the plotted time course (s)
SMOOTH_S = 60  # halflife (s) of the smoothing of the plotted mean line; None = unsmoothed
TICK_STEP = 600  # x tick spacing in seconds
XLIM = (0, 3600)  # x range of the time-course plot, seconds relative to ALIGN_EVENT
YLIM = None  # y range of the time-course plot, e.g. (0.5, 4); None = automatic
FIGSIZE_LINE = (4, 3)  # time-course figure size in inches
WINDOW = (1800, 3350)  # summary window in seconds relative to ALIGN_EVENT (lower edge included, upper excluded)
CENTER = "mean"  # per-animal statistic in WINDOW and its plotted centre: "mean" or "median"
FIGSIZE_STRIP = (2.5, 3.5)  # per-animal summary figure size in inches; (3.8, 3.5) fits three conditions
P_ADJUST = "holm"  # correction of the post-hoc paired t-tests across condition pairs ("holm", "bonferroni" or "fdr_bh")
COLORS = {"AwakeControl": "gray", "No Induction": "#E377C2", "Induction": "#5F4EAB"}  # colour per condition label (add your own labels)
OVERWRITE_RAW = False  # True re-extracts the raw CSVs
OVERWRITE_BANDS = False  # True recomputes the band CSVs (needed after changing BAND_DEFINITIONS for an existing band)
WORK_DIR = "cache"  # processing cache (raw and band CSVs), reused between runs
OUTPUT_DIR = "outputs/06_combine_across_animals"  # figures and tables; files are never overwritten
FIGURE_FORMATS = ("png", "svg")  # one file per format for each figure
DPI = 300  # resolution of the png

In [ ]:
from pathlib import Path  # path handling
import sys  # module search path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ecog").is_dir())  # repo root = first parent folder containing ecog/
sys.path.insert(0, str(ROOT))  # makes `import ecog` work without installing anything
import os  # path normalising
EDF_DIR = Path(os.path.abspath(EDF_FOLDER))  # absolute EDF folder with any `..` collapsed (relative paths start at this notebook's folder)
import mne  # EDF reader used by ecog
mne.set_log_level("WARNING")  # hides the EDF reader's progress messages
import pandas as pd  # tables
from ecog import (Recording, extractRecording, computeBandRatios, aggregateAligned, compareConditionsANOVA, compareConditionMeans,
                  BAND_RATIOS, saveFigure, saveTable, writeRunInfo)  # processing, aggregation, statistics, saving
from ecog.plots import plotRatioSNR, plot_condition_stripplot  # aligned time courses and per-animal summary
WORK = Path(WORK_DIR)  # cache folder
OUT = Path(OUTPUT_DIR)  # output folder

### 1. Recordings grouped by condition

In [ ]:
def toRecording(entry):  # (RECORDINGS index or dict, channel) -> (Recording, channel)
    spec, channel = entry  # recording description and the channel analysed
    spec = RECORDINGS[spec] if isinstance(spec, int) else spec  # an int picks from RECORDINGS, a dict is used as given
    return Recording(edf_folder=EDF_DIR, **{**spec, "channel_index": channel}), channel  # the channel in CONDITIONS decides what is extracted


# {condition label: [(Recording, channel), ...]}, the input of every function below
batches = {label: [toRecording(e) for e in entries] for label, entries in CONDITIONS.items()}
ORDER = list(CONDITIONS)  # condition order in figures and tests
for entries in batches.values():  # check every file, channel and event before processing
    for rec, _ in entries:
        rec.validate()  # stops with one message listing every problem
{n: len(v) for n, v in batches.items()}  # recordings per condition

### 2. EDF to raw CSV and band SNR CSV (each skipped when cached)

In [ ]:
DEFINITIONS = {**BAND_RATIOS, **BAND_DEFINITIONS}  # band SNR definitions in use
for entries in batches.values():  # every recording of every condition
    for rec, _ in entries:
        extractRecording(rec, WORK, overwrite=OVERWRITE_RAW)  # raw CSV of the channel
        computeBandRatios(rec, WORK, [BAND], definitions=DEFINITIONS, overwrite=OVERWRITE_BANDS)  # band SNR CSV

### 3. Aligned long table: one row per time segment, mouse and condition

In [ ]:
# each condition's recordings stacked on a shared axis (0 = ALIGN_EVENT); mouse = <animal>_R<session>
aggregated = pd.concat([aggregateAligned(entries, WORK, BAND, "ratio", align_event=ALIGN_EVENT, condition_label=label)
                        for label, entries in batches.items()], ignore_index=True)
path = saveTable(aggregated, OUT / f"{BAND}_snr_aligned.csv")  # aggregate table, never overwritten
print(path, aggregated.shape)  # file written and table size

### 4. Mean +/- SEM time course per condition (animals averaged within each bin)

In [ ]:
# per animal and bin the median, then mean +/- SEM across animals
ax = plotRatioSNR(aggregated, ylabel=f"{BAND} SNR (mean +/- SEM)", events={ALIGN_EVENT: 0}, colors={ALIGN_EVENT: "black"},
                  binning_s=BIN_S, smoothing_s=SMOOTH_S, tick_step=TICK_STEP, line_colors=COLORS, error_metric="sem", figsize=FIGSIZE_LINE)
ax.set_xlim(*XLIM)  # x range in seconds (tick labels are minutes)
ax.set_ylim(YLIM)  # y range (None = automatic)
saveFigure(ax.figure, OUT / f"{BAND}_snr_timecourse", dpi=DPI, formats=FIGURE_FORMATS)  # saved without overwriting

### 5. How animals are matched: the animal id is the mouse id with its _R<n> suffix removed

In [ ]:
animal = aggregated["mouse"].str.replace(r"_R\d+$", "", regex=True)  # animal id used by every statistic below
sessions = aggregated.assign(animal=animal).groupby(["animal", "condition"])["mouse"].nunique().unstack()[ORDER]  # sessions per animal and condition
sessions  # a value above 1 = two sessions of one animal in one condition

Two sessions of one animal in one condition: the window summary (step 6) and the tests (step 7) pool every segment of both sessions inside `WINDOW` into that animal's single value (weighted by segment count), so the animal is counted once. The time course (step 4) averages per mouse = session, so the legend `n` counts sessions and the second session adds weight there. To try it, add `(12, 0)` (a second PEEG01M Induction session; its 2026-08-26 .edf must be in `EDF_FOLDER`) to the Induction list of `CONDITIONS` and rerun from step 1. With two conditions the ANOVA F equals the paired t squared; a third condition (`No Induction`) makes the post-hoc `p_adjust` informative.

### 6. Per-animal window summary (one point per animal, gray lines join each animal across conditions)

In [ ]:
# per-animal mean (or median) in WINDOW, centre +/- spread per condition
ax, summary = plot_condition_stripplot(aggregated, "ratio", WINDOW, ORDER, ylabel=f"{BAND} SNR", title=f"{WINDOW[0] // 60}-{WINDOW[1] // 60} min",
                                       figsize=FIGSIZE_STRIP, box_color=COLORS, center=CENTER, line_color="gray")
saveFigure(ax.figure, OUT / f"{BAND}_snr_window_summary", dpi=DPI, formats=FIGURE_FORMATS)  # saved without overwriting
summary  # per-condition mean, median, std, count, mad

### 7. Repeated-measures ANOVA across conditions, then post-hoc paired t-tests

In [ ]:
# per-animal values in WINDOW, one-way RM-ANOVA; included/excluded lists name animals with/without a value in every condition
anovaTable, included, excluded = compareConditionsANOVA(aggregated, "ratio", WINDOW, ORDER, center=CENTER)
# paired t-test for every condition pair with a multiple-comparison correction in column p_adj
posthoc = compareConditionMeans(aggregated, "ratio", WINDOW, ORDER, p_adjust=P_ADJUST, center=CENTER)
print(anovaTable.round(4), "\nincluded:", included, "excluded:", excluded)  # omnibus test and the animals it used
posthoc.round(4)  # post-hoc table

### 8. Save tables

In [ ]:
lo, hi = WINDOW  # window edges
perAnimal = (aggregated[(aggregated["timeSeg"] >= lo) & (aggregated["timeSeg"] < hi)].assign(animal=animal)
             .groupby(["animal", "condition"])["ratio"].agg(CENTER).unstack()[ORDER])  # the per-animal values behind plot and tests
paths = [saveTable(perAnimal.reset_index(), OUT / f"{BAND}_snr_per_animal_window.csv"),
         saveTable(anovaTable.reset_index(), OUT / f"{BAND}_snr_anova.csv"),
         saveTable(posthoc, OUT / f"{BAND}_snr_posthoc.csv")]  # window values and statistics, never overwritten
info = writeRunInfo(OUT, settings=dict(conditions={k: len(v) for k, v in CONDITIONS.items()}, band=BAND, align_event=ALIGN_EVENT, window=WINDOW,
                                       center=CENTER, p_adjust=P_ADJUST, band_definitions=BAND_DEFINITIONS))  # versions and settings
print(paths, info)  # files written